# 📓 The GraphRAG Bakeoff

---

## The Core Concept: Why Vector Search is Not Enough

As Data Engineers, we are used to **Exact Matches** (`WHERE id = 5`). In AI, we use **Semantic Matches** (Vectors).

* **VectorRAG (Semantic Entry):** Converts text into a list of numbers (coordinates). It finds the "nearest" piece of text. But it is **blind** to structure.
* **GraphRAG (Structural Reasoning):** Once we find the starting point via Vector Search, we follow the **edges** (relationships) to find context that is logically connected but semantically different.

In [ ]:
import os
from openai import OpenAI
from arango import ArangoClient
from dotenv import load_dotenv

load_dotenv()

# Use environment variables for credentials
ARANGO_URL = os.getenv("ARANGO_URL")
ARANGO_PWD = os.getenv("ARANGO_PASSWORD") # Maps to ARANGO_PASSWORD in .env
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)
arango_client = ArangoClient(hosts=ARANGO_URL)
db = arango_client.db('glass_box', username='root', password=ARANGO_PWD)

def get_embedding(text):
    return client.embeddings.create(
        input=[text], 
        model="text-embedding-3-small"
    ).data[0].embedding

## Step 3: The Idempotent Search View
We create a **View**, which is a specialized index that allows ArangoDB to perform "Fuzzy" Vector searches.

In [ ]:
# ---------------------------------------------------------
# STEP 3: CREATING THE SEMANTIC LANDING PAD
# ---------------------------------------------------------
VIEW_NAME = 'kb_view'
existing_views = [v['name'] for v in db.views()]
if VIEW_NAME in existing_views:
    db.delete_view(VIEW_NAME)

# Note the specific structure: we define the 'vector' type WITHIN the fields.
db.create_view(VIEW_NAME, 'arangosearch', {
    'links': {
        'kb_nodes': {
            'fields': {
                'embedding': {
                    'index': True,
                    'type': 'vector',
                    'dimension': 1536, # OpenAI text-embedding-3-small
                    'distance_metric': 'cosine'
                }
            }
        }
    }
})
print("✅ Vector Index established with native 'vector' type.")

✅ Vector Index established with native 'vector' type.


## Step 4: The Bakeoff Logic
### Understanding the AQL Syntax for Data Engineers:
1. **`@vec`**: This is a **Bind Parameter**. In the Python code below, you'll see `bind_vars={'vec': query_vector}`. This is how we securely pass our question's coordinates into the database engine.
2.  **`SEARCH APPROX_NEAR(...)`**: This is your "Fuzzy Filter." It replaces `WHERE`. It looks at the `doc.embedding` (the coordinates stored in the DB) and compares them to `@vec` (the coordinates of your question).
3.  **`SORT BM25(doc) DESC`**: BM25 is a classic search ranking algorithm. It ensures that if we have multiple matches, the most relevant text rises to the top.
4.  **`FOR v, e IN 1..3 OUTBOUND anchor kb_edges`**: This is a **Graph Traversal**. 
    * `1..3` means "Follow the links up to 3 steps away."
    * `OUTBOUND` means follow the direction of the arrows.
    * `v` is the **Vertex** (the destination node).
    * `e` is the **Edge** (the relationship itself).
5.  **`doc.embedding` vs `anchor.embedding`**: These are just aliases. In the Vector query, we call it `doc` (short for document). In the Graph query, we call it `anchor` because it's the "anchor point" where our traversal begins.

In [ ]:
query = "What security constraints are linked to Sreeram's work?"
query_vector = get_embedding(query)

print(f"Generated Embedding for query. Length: {len(query_vector)}")

# PATH A: Standard Vector RAG
vector_aql = """
FOR doc IN kb_view
  SEARCH APPROX_NEAR(doc.embedding, @vec, "cosine")
  SORT BM25(doc) DESC
  LIMIT 2
  RETURN doc.text
"""

# PATH B: GraphRAG
graph_aql = """
FOR anchor IN kb_view
  SEARCH APPROX_NEAR(anchor.embedding, @vec, "cosine")
  LIMIT 1
  FOR v, e IN 1..3 OUTBOUND anchor kb_edges
    RETURN { 
        text: v.text, 
        relationship: e.label 
    }
"""

# Execute with bind_vars to bridge Python and AQL
v_context = list(db.aql.execute(vector_aql, bind_vars={'vec': query_vector}))
g_context = list(db.aql.execute(graph_aql, bind_vars={'vec': query_vector}))

print("Bakeoff complete.")

Generated Embedding for query. Length: 1536


AQLQueryExecuteError: [HTTP 400][ERR 1540] usage of unknown function 'APPROX_NEAR()'

## Visualizing the 'Reasoning Gap'
Watch how Vector RAG only sees the 'person', while GraphRAG finds the 'protocol'.

In [ ]:
print(f"--- [VECTOR RAG CONTEXT] ---\n{v_context}")

print(f"\n--- [GRAPHRAG CONTEXT] ---")
for item in g_context:
    print(f"[{item['relationship']}] -> {item['text']}")

In [ ]:
def ask_llm(context):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Answer based ONLY on context."},
            {"role": "user", "content": f"Query: {query}\n\nContext: {context}"}
        ]
    )
    return response.choices[0].message.content

print(f"Vector Answer: {ask_llm(v_context)}")
print(f"GraphRAG Answer: {ask_llm(g_context)}")